In [ ]:
import duckdb

# Adapter le chemin vers ton fichier décompressé
filepath = "../data/raw/AIS_2017_08_01.csv"
# Aperçu des 5 premières lignes
con = duckdb.connect()
print(con.sql(f"SELECT * FROM read_csv_auto('{filepath}') LIMIT 5").show())

# Nombre de lignes et colonnes
print(con.sql(f"SELECT COUNT(*) as nb_lignes FROM read_csv_auto('{filepath}')").show())

# Noms des colonnes
print(con.sql(f"DESCRIBE SELECT * FROM read_csv_auto('{filepath}')").show())

In [ ]:
import duckdb

con = duckdb.connect()

# Bounding box Houston Ship Channel + Port of Houston
# Le chenal va de Galveston Bay jusqu'au port intérieur
LAT_MIN, LAT_MAX = 29.3, 29.85
LON_MIN, LON_MAX = -95.4, -94.7

# Filtre zone Houston + export Parquet
con.sql(f"""
    COPY (
        SELECT *
        FROM read_csv_auto('../data/raw/AIS_2017_08_01.csv')
        WHERE LAT BETWEEN {LAT_MIN} AND {LAT_MAX}
          AND LON BETWEEN {LON_MIN} AND {LON_MAX}
    ) TO 'houston_2017_08_01.parquet' (FORMAT PARQUET)
""")

# Vérifie le résultat
result = con.sql("SELECT COUNT(*) as nb, COUNT(DISTINCT MMSI) as nb_navires FROM 'houston_2017_08_01.parquet'")
print(result.show())

# Aperçu de la distribution SOG (navires à l'arrêt vs en mouvement)
print(con.sql("""
    SELECT 
        CASE 
            WHEN SOG < 0.5 THEN 'A l arret (<0.5 noeud)'
            WHEN SOG < 5 THEN 'Lent (0.5-5 noeuds)'
            WHEN SOG < 10 THEN 'Moyen (5-10 noeuds)'
            ELSE 'Rapide (>10 noeuds)'
        END as categorie_vitesse,
        COUNT(*) as nb_messages,
        COUNT(DISTINCT MMSI) as nb_navires
    FROM 'houston_2017_08_01.parquet'
    GROUP BY 1
    ORDER BY 2 DESC
""").show())

In [ ]:
import duckdb
import polars as pl
import numpy as np
import hdbscan
import matplotlib.pyplot as plt

con = duckdb.connect()

# UNE position moyenne par navire (au lieu de 1440)
df_static = con.sql("""
    SELECT 
        MMSI,
        AVG(LAT) as LAT, 
        AVG(LON) as LON,
        COUNT(*) as nb_messages,
        MAX(Draft) as Draft,
        MAX(VesselType) as VesselType,
        MAX(Length) as Length
    FROM 'houston_2017_08_01.parquet'
    WHERE SOG < 0.5
    GROUP BY MMSI
""").pl()

print(f"Navires à l'arrêt : {len(df_static)}")

# HDBSCAN sur les positions moyennes
coords = df_static.select(['LAT', 'LON']).to_numpy()
coords_rad = np.radians(coords)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    metric='haversine',
    cluster_selection_method='eom'
)

labels = clusterer.fit_predict(coords_rad)
probabilities = clusterer.probabilities_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = (labels == -1).sum()
print(f"Clusters détectés : {n_clusters}")
print(f"Navires bruit : {n_noise} ({100*n_noise/len(labels):.1f}%)")

df_static = df_static.with_columns([
    pl.Series("cluster", labels),
    pl.Series("membership_score", probabilities)
])

# Top 15 clusters par nombre de navires
for c in sorted(set(labels)):
    if c == -1:
        continue
    mask = labels == c
    n_vessels = mask.sum()
    if n_vessels < 3:
        continue
    lat_mean = coords[mask, 0].mean()
    lon_mean = coords[mask, 1].mean()
    avg_member = probabilities[mask].mean()
    print(f"  Cluster {c}: {n_vessels} navires, "
          f"centre=({lat_mean:.4f}, {lon_mean:.4f}), "
          f"membership={avg_member:.2f}")

In [ ]:
import folium

m = folium.Map(location=[29.6, -95.0], zoom_start=11, tiles='CartoDB positron')

colors = ['red', 'blue', 'green', 'orange', 'purple', 'darkred', 
          'cadetblue', 'darkgreen', 'darkblue', 'pink']

cluster_sizes = {}
for c in sorted(set(labels)):
    if c == -1:
        continue
    cluster_sizes[c] = (labels == c).sum()

top_clusters = sorted(cluster_sizes, key=cluster_sizes.get, reverse=True)[:15]

for i, c in enumerate(top_clusters):
    mask = labels == c
    color = colors[i % len(colors)]
    n_vessels = mask.sum()
    lat_mean = coords[mask, 0].mean()
    lon_mean = coords[mask, 1].mean()
    
    lats = coords[mask, 0]
    lons = coords[mask, 1]
    for lat, lon in zip(lats, lons):
        folium.CircleMarker(
            [lat, lon], radius=4, color=color, 
            fill=True, fillOpacity=0.7,
            popup=f"Cluster {c} ({n_vessels} navires)"
        ).add_to(m)
    
    folium.Marker(
        [lat_mean, lon_mean],
        icon=folium.DivIcon(html=f'<b style="color:{color};font-size:10px">C{c}: {n_vessels}nav</b>')
    ).add_to(m)

noise_mask = labels == -1
for lat, lon in zip(coords[noise_mask, 0], coords[noise_mask, 1]):
    folium.CircleMarker(
        [lat, lon], radius=2, color='gray', 
        fill=True, fillOpacity=0.3
    ).add_to(m)

m.save('houston_clusters_map.html')
print("Carte sauvegardée : houston_clusters_map.html")